# 03 — Noisy y baseline: repo TPE

Цель: проверить, как базовые алгоритмы ведут себя при шуме в значениях функции.

Сравниваются:

- `random_clean`
- `no_w_clean`
- `optuna_clean`
- `random_noisy_y`
- `no_w_noisy_y`
- `optuna_noisy_y`

Важно:

- при `noisy_y` алгоритм оптимизирует зашумлённую функцию;
- итоговое качество всегда оценивается по **clean raw function**;
- \(w(x)\), градиенты и нормализация здесь не используются.


In [ ]:
# ============================================================
# ЯЧЕЙКА 1: Установка зависимостей
# ============================================================
!pip uninstall -y ConfigSpace
!pip install -U "ConfigSpace==1.2.0"
!pip -q install parzen-estimator numpy matplotlib optuna

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.8/130.8 kB 5.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for ConfigSpace: filename=configspace-1.2.0-py3-none-any.whl size=115899 sha256=91359de1f26720553e1d65392e35abbf17eca0e17306e6600e50f739a240bad1
  Stored in directory: /root/.cache/pip/wheels/1c/c1/6b/8c7b7a188c6753c0b2e2fca5cfe7dd7d1e2fbb52eed0e54c75
Successfully built ConfigSpace
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 13.8 MB/s eta 0:00:00


In [ ]:
# ============================================================
# 1. Imports + repo path
# Блок повторяет рабочую схему imports из твоего notebook.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import time
from pathlib import Path
import sys
import pandas as pd
import math

import ConfigSpace as CS
import ConfigSpace.hyperparameters as CSH

try:
    import optuna
    from optuna.samplers import TPESampler
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    HAS_OPTUNA = True
except Exception as e:
    HAS_OPTUNA = False
    print("Optuna не импортировалась:", repr(e))

# ------------------------------------------------------------
# ConfigSpace q-patch
# ------------------------------------------------------------

def _q_prop(self):
    return getattr(self, "_q", None)

_NEED_Q = [
    "UniformFloatHyperparameter",
    "UniformIntegerHyperparameter",
    "NormalFloatHyperparameter",
    "NormalIntegerHyperparameter",
]

for name in _NEED_Q:
    cls = getattr(CSH, name, None)
    if cls is not None and not hasattr(cls, "q"):
        setattr(cls, "q", property(_q_prop))

# ------------------------------------------------------------
# Google Drive
# ------------------------------------------------------------

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive mount skipped:", repr(e))

# ------------------------------------------------------------
# Repo path
# ВАЖНО: добавляем только корень, где лежит папка tpe.
# ------------------------------------------------------------

TPE_ROOT = Path("drive/MyDrive/content")

if not (TPE_ROOT / "tpe").exists():
    alt = Path("/content/drive/MyDrive/content")
    if (alt / "tpe").exists():
        TPE_ROOT = alt

if not (TPE_ROOT / "tpe").exists():
    raise FileNotFoundError(
        f"Не найдена папка repo: {TPE_ROOT / 'tpe'}"
    )

if str(TPE_ROOT) not in sys.path:
    sys.path.append(str(TPE_ROOT))

# Чистим возможный сломанный импорт из прошлого запуска kernel
for key in list(sys.modules.keys()):
    if key == "tpe" or key.startswith("tpe."):
        del sys.modules[key]

(TPE_ROOT / "tpe" / "__init__.py").touch(exist_ok=True)

from tpe.optimizer.tpe_optimizer import TPEOptimizer
from tpe.optimizer.random_search import RandomSearch

print("TPE_ROOT =", TPE_ROOT)
print("Repo imports OK")
print("HAS_OPTUNA =", HAS_OPTUNA)


Mounted at /content/drive
TPE_ROOT = drive/MyDrive/content
Repo imports OK
HAS_OPTUNA = True


# 2. Clean raw functions

Clean-функции используются для финальной оценки качества во всех вариантах.


In [ ]:
def sphere_raw(x):
    x = np.asarray(x, dtype=float)
    return float(np.sum(x ** 2))


def rosenbrock_raw(x):
    x = np.asarray(x, dtype=float)
    return float(
        np.sum(
            100.0 * (x[1:] - x[:-1] ** 2) ** 2
            + (1.0 - x[:-1]) ** 2
        )
    )


def rastrigin_raw(x):
    x = np.asarray(x, dtype=float)
    d = len(x)
    return float(
        10.0 * d + np.sum(x ** 2 - 10.0 * np.cos(2.0 * np.pi * x))
    )


def ackley_raw(x):
    x = np.asarray(x, dtype=float)
    d = len(x)
    a = 20.0
    b = 0.2
    c = 2.0 * np.pi
    s1 = np.sum(x ** 2)
    s2 = np.sum(np.cos(c * x))
    return float(
        -a * np.exp(-b * np.sqrt(s1 / d))
        - np.exp(s2 / d)
        + a
        + np.e
    )


CLEAN_MAP = {
    "Sphere(2D)": sphere_raw,
    "Rosenbrock(2D)": rosenbrock_raw,
    "Rastrigin(2D)": rastrigin_raw,
    "Ackley(2D)": ackley_raw,
}

TRUE_MINIMA = {
    "Sphere(2D)": np.array([0.0, 0.0]),
    "Rosenbrock(2D)": np.array([1.0, 1.0]),
    "Rastrigin(2D)": np.array([0.0, 0.0]),
    "Ackley(2D)": np.array([0.0, 0.0]),
}


# 3. Noisy y functions

Шум добавляется только к наблюдаемому значению:

\[
f_{\text{obs}}(x)=f_{\text{clean}}(x)+\varepsilon,\quad \varepsilon\sim \mathcal N(0,\sigma^2).
\]

Для воспроизводимости шум зависит от `seed`, `fn_name` и номера вызова.


In [ ]:
NOISE_SIGMA = {
    "Sphere(2D)":     0.005,
    "Rosenbrock(2D)": 0.05,
    "Rastrigin(2D)":  0.25,
    "Ackley(2D)":     0.05,
}


def make_noisy_y_fn(fn_name, seed):
    clean_fn = CLEAN_MAP[fn_name]
    sigma = NOISE_SIGMA[fn_name]
    rng = np.random.default_rng(10_000 + 997 * seed + abs(hash(fn_name)) % 10_000)

    def noisy_fn(x):
        return float(clean_fn(x) + rng.normal(0.0, sigma))

    return noisy_fn


# 4. ConfigSpace

In [ ]:
def make_cs(bounds, seed=0):
    cs = CS.ConfigurationSpace(seed=seed)
    hps = [
        CSH.UniformFloatHyperparameter(name, lower=lo, upper=hi)
        for name, lo, hi in bounds
    ]
    cs.add_hyperparameters(hps)
    return cs


CS_MAP = {
    "Sphere(2D)":     make_cs([("x0", -5.0, 5.0), ("x1", -5.0, 5.0)], seed=0),
    "Rosenbrock(2D)": make_cs([("x0", -2.0, 2.0), ("x1", -1.0, 3.0)], seed=0),
    "Rastrigin(2D)":  make_cs([("x0", -5.12, 5.12), ("x1", -5.12, 5.12)], seed=0),
    "Ackley(2D)":     make_cs([("x0", -5.0, 5.0), ("x1", -5.0, 5.0)], seed=0),
}

BOUNDS_MAP = {
    "Sphere(2D)":     [(-5.0, 5.0), (-5.0, 5.0)],
    "Rosenbrock(2D)": [(-2.0, 2.0), (-1.0, 3.0)],
    "Rastrigin(2D)":  [(-5.12, 5.12), (-5.12, 5.12)],
    "Ackley(2D)":     [(-5.0, 5.0), (-5.0, 5.0)],
}


# 5. Experiment settings

In [ ]:
SEEDS = tuple(range(30))

N_INIT = 10
MAX_EVALS = 100

BASE_KWARGS = dict(
    n_ei_candidates=24,
    min_bandwidth_factor=1e-2,
    top=0.2,
)

# Thresholds считаются по clean raw objective
THRESHOLDS = {
    "Sphere(2D)":     1e-2,
    "Rosenbrock(2D)": 1e-1,
    "Rastrigin(2D)":  2.0,
    "Ackley(2D)":     0.5,
}

RESULTS_ROOT = Path(".tpe_results_03_noisy_y")
RESULTS_ROOT.mkdir(exist_ok=True)

print("SEEDS =", len(SEEDS))
print("MAX_EVALS =", MAX_EVALS)
print("NOISE_SIGMA =", NOISE_SIGMA)


SEEDS = 30
MAX_EVALS = 100
NOISE_SIGMA = {'Sphere(2D)': 0.005, 'Rosenbrock(2D)': 0.05, 'Rastrigin(2D)': 0.25, 'Ackley(2D)': 0.05}


# 6. Runner

`objective_fn` — то, что видит алгоритм.  
`clean_fn` — то, по чему оценивается итоговое качество.

Для `noisy_y` алгоритм видит зашумлённый loss, но final metrics считаются по clean loss.


In [ ]:
def running_best(losses):
    best, cur = [], float("inf")
    for v in losses:
        cur = min(cur, float(v))
        best.append(cur)
    return np.array(best)


def run_once(
    OptClass,
    objective_fn,
    clean_fn,
    cs,
    *,
    seed,
    n_init,
    max_evals,
    tpe_kwargs=None,
    fn_name,
    data_type,
):
    tpe_kwargs = tpe_kwargs or {}

    algo = "tpe" if OptClass is TPEOptimizer else "random"

    out_dir = RESULTS_ROOT / fn_name / data_type
    out_dir.mkdir(parents=True, exist_ok=True)

    resultfile = str(out_dir / f"{algo}_seed{seed}")

    def obj_func_instance(cfg):
        x = np.array([cfg["x0"], cfg["x1"]], dtype=float)
        return {"loss": float(objective_fn(x))}, 0.0

    if OptClass is TPEOptimizer:
        opt = OptClass(
            obj_func=obj_func_instance,
            config_space=cs,
            resultfile=resultfile,
            n_init=n_init,
            max_evals=max_evals,
            seed=seed,
            metric_name="loss",
            **tpe_kwargs,
        )
    else:
        opt = OptClass(
            obj_func=obj_func_instance,
            config_space=cs,
            resultfile=resultfile,
            n_init=n_init,
            max_evals=max_evals,
            seed=seed,
            metric_name="loss",
        )

    best_cfg, best_loss = opt.optimize()
    obs = opt.fetch_observations()

    observed_losses = np.asarray(obs["loss"], dtype=float)

    true_min = TRUE_MINIMA[fn_name]

    best_observed = float("inf")
    best_x = None

    clean_curve = []
    x_distance_curve = []

    for i in range(len(observed_losses)):
        x = np.array([obs["x0"][i], obs["x1"][i]], dtype=float)

        if observed_losses[i] < best_observed:
            best_observed = float(observed_losses[i])
            best_x = x

        clean_curve.append(float(clean_fn(best_x)))
        x_distance_curve.append(float(np.linalg.norm(best_x - true_min)))

    return {
        "best_cfg": best_cfg,
        "best_loss_observed": float(best_loss),
        "observed_curve": running_best(observed_losses),
        "clean_curve": np.array(clean_curve),
        "x_distance_curve": np.array(x_distance_curve),
        "x_history": np.array([[obs["x0"][i], obs["x1"][i]] for i in range(len(observed_losses))], dtype=float),
    }


def run_benchmark_for_data(fn_name, data_type, seeds):
    clean_fn = CLEAN_MAP[fn_name]
    cs = CS_MAP[fn_name]

    out = {
        f"random_{data_type}": [],
        f"no_w_{data_type}": [],
    }

    for s in seeds:
        if data_type == "clean":
            objective_fn_random = clean_fn
            objective_fn_tpe = clean_fn
        elif data_type == "noisy_y":
            # отдельные шумовые генераторы для методов, чтобы вызовы не зависели друг от друга
            objective_fn_random = make_noisy_y_fn(fn_name, seed=10_000 + s)
            objective_fn_tpe = make_noisy_y_fn(fn_name, seed=20_000 + s)
        else:
            raise ValueError(data_type)

        out[f"random_{data_type}"].append(
            run_once(
                RandomSearch,
                objective_fn_random,
                clean_fn,
                cs,
                seed=s,
                n_init=N_INIT,
                max_evals=MAX_EVALS,
                fn_name=fn_name,
                data_type=data_type,
            )
        )

        out[f"no_w_{data_type}"].append(
            run_once(
                TPEOptimizer,
                objective_fn_tpe,
                clean_fn,
                cs,
                seed=s,
                n_init=N_INIT,
                max_evals=MAX_EVALS,
                tpe_kwargs=BASE_KWARGS,
                fn_name=fn_name,
                data_type=data_type,
            )
        )

    return out


# 7. Optuna reference

In [ ]:
def run_optuna_tpe(fn_name, data_type, seed):
    if not HAS_OPTUNA:
        return None

    clean_fn = CLEAN_MAP[fn_name]
    true_min = TRUE_MINIMA[fn_name]
    bounds = BOUNDS_MAP[fn_name]

    if data_type == "clean":
        objective_fn = clean_fn
    elif data_type == "noisy_y":
        objective_fn = make_noisy_y_fn(fn_name, seed=30_000 + seed)
    else:
        raise ValueError(data_type)

    sampler = TPESampler(
        seed=seed,
        n_startup_trials=N_INIT,
        n_ei_candidates=24,
    )

    study = optuna.create_study(direction="minimize", sampler=sampler)

    observed_losses = []
    clean_curve = []
    x_distance_curve = []

    best_observed = float("inf")
    best_x = None

    def objective(trial):
        nonlocal best_observed, best_x

        x0 = trial.suggest_float("x0", bounds[0][0], bounds[0][1])
        x1 = trial.suggest_float("x1", bounds[1][0], bounds[1][1])
        x = np.array([x0, x1], dtype=float)

        loss = float(objective_fn(x))
        observed_losses.append(loss)

        if loss < best_observed:
            best_observed = loss
            best_x = x

        clean_curve.append(float(clean_fn(best_x)))
        x_distance_curve.append(float(np.linalg.norm(best_x - true_min)))

        return loss

    study.optimize(objective, n_trials=MAX_EVALS, show_progress_bar=False)

    observed_losses = np.array(observed_losses, dtype=float)

    return {
        "best_loss_observed": float(study.best_value),
        "observed_curve": running_best(observed_losses),
        "clean_curve": np.array(clean_curve),
        "x_distance_curve": np.array(x_distance_curve),
        "x_history": np.array([[t.params["x0"], t.params["x1"]] for t in study.trials], dtype=float),
    }


# 8. Run experiment

In [ ]:
all_results = {}

start_time = time.time()

for fn_name in CLEAN_MAP.keys():
    print(f"\n{'=' * 60}")
    print(fn_name)
    print(f"{'=' * 60}")

    all_results[fn_name] = {}

    for data_type in ["clean", "noisy_y"]:
        print("data:", data_type)

        res = run_benchmark_for_data(fn_name, data_type, SEEDS)
        all_results[fn_name].update(res)

        if HAS_OPTUNA:
            all_results[fn_name][f"optuna_{data_type}"] = [
                run_optuna_tpe(fn_name, data_type, seed=s)
                for s in SEEDS
            ]

elapsed = time.time() - start_time
print(f"\nDone. Time: {elapsed:.2f} sec")



Sphere(2D)
data: clean
data: noisy_y

Rosenbrock(2D)
data: clean
data: noisy_y

Rastrigin(2D)
data: clean
data: noisy_y

Ackley(2D)
data: clean
data: noisy_y

Done. Time: 119.42 sec


# 9. Summary table

In [ ]:
rows = []

for fn_name, variants in all_results.items():
    threshold = THRESHOLDS[fn_name]

    for variant, runs in variants.items():
        steps_list = []
        clean_final = []
        dist_final = []

        for r in runs:
            step = next(
                (i + 1 for i, v in enumerate(r["clean_curve"]) if v <= threshold),
                None,
            )

            steps_list.append(step)
            clean_final.append(float(r["clean_curve"][-1]))
            dist_final.append(float(r["x_distance_curve"][-1]))

        successful_steps = [s for s in steps_list if s is not None]

        method = (
            variant
            .replace("_clean", "")
            .replace("_noisy_y", "")
        )

        data_type = "noisy_y" if variant.endswith("_noisy_y") else "clean"

        row = {
            "function": fn_name,
            "variant": variant,
            "method": method,
            "data_type": data_type,
            "noise_sigma": NOISE_SIGMA[fn_name] if data_type == "noisy_y" else 0.0,
            "mean_steps": np.mean(successful_steps) if successful_steps else np.nan,
            "success_rate_%": 100.0 * len(successful_steps) / len(steps_list),
            "clean_mean": np.mean(clean_final),
            "clean_median": np.median(clean_final),
            "clean_std": np.std(clean_final),
            "dist_mean": np.mean(dist_final),
            "dist_median": np.median(dist_final),
            "dist_std": np.std(dist_final),
        }

        for i, s in enumerate(steps_list):
            row[f"seed{i}"] = s

        rows.append(row)

seed_cols = [f"seed{i}" for i in range(len(SEEDS))]

base_cols = [
    "function",
    "variant",
    "method",
    "data_type",
    "noise_sigma",
    "mean_steps",
    "success_rate_%",
    "clean_mean",
    "clean_median",
    "clean_std",
    "dist_mean",
    "dist_median",
    "dist_std",
]

df_summary = pd.DataFrame(rows)[base_cols + seed_cols]
df_summary = df_summary.sort_values(["function", "method", "data_type"])

df_summary


,function,variant,method,data_type,noise_sigma,mean_steps,success_rate_%,clean_mean,clean_median,clean_std,...,seed20,seed21,seed22,seed23,seed24,seed25,seed26,seed27,seed28,seed29
19,Ackley(2D),no_w_clean,no_w,clean,0.000,58.166667,40.000000,1.690338,2.581500,1.425146,...,NaN,42.0,NaN,NaN,67.0,54.0,NaN,NaN,NaN,51.0
22,Ackley(2D),no_w_noisy_y,no_w,noisy_y,0.050,58.625000,26.666667,1.983596,2.587158,1.274092,...,NaN,53.0,NaN,NaN,70.0,NaN,NaN,NaN,98.0,37.0
20,Ackley(2D),optuna_clean,optuna,clean,0.000,59.466667,50.000000,0.601988,0.501673,0.445187,...,NaN,NaN,NaN,23.0,32.0,NaN,84.0,84.0,45.0,NaN
23,Ackley(2D),optuna_noisy_y,optuna,noisy_y,0.050,60.785714,46.666667,0.648889,0.548643,0.570642,...,NaN,NaN,NaN,23.0,77.0,NaN,84.0,NaN,38.0,NaN
18,Ackley(2D),random_clean,random,clean,0.000,56.000000,3.333333,2.843320,2.909938,1.035302,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21,Ackley(2D),random_noisy_y,random,noisy_y,0.050,56.000000,3.333333,2.843756,2.909938,1.035772,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13,Rastrigin(2D),no_w_clean,no_w,clean,0.000,45.636364,36.666667,3.373986,2.060831,2.625678,...,75.0,NaN,NaN,NaN,NaN,NaN,NaN,27.0,17.0,NaN
16,Rastrigin(2D),no_w_noisy_y,no_w,noisy_y,0.250,44.636364,36.666667,3.888182,4.176209,2.779351,...,41.0,NaN,NaN,76.0,NaN,NaN,NaN,27.0,17.0,NaN
14,Rastrigin(2D),optuna_clean,optuna,clean,0.000,60.461538,43.333333,2.165251,2.072262,1.085757,...,22.0,77.0,NaN,60.0,NaN,NaN,NaN,NaN,82.0,NaN
17,Rastrigin(2D),optuna_noisy_y,optuna,noisy_y,0.250,60.214286,46.666667,2.102730,2.090531,1.232608,...,22.0,77.0,NaN,60.0,NaN,100.0,NaN,72.0,82.0,NaN


In [ ]:
out_dir = Path("results")
out_dir.mkdir(exist_ok=True)

summary_path = out_dir / "03_noisy_y_baseline_summary.csv"
df_summary.to_csv(summary_path, index=False)

print("Saved:", summary_path)


Saved: results/03_noisy_y_baseline_summary.csv


# 10. Plots

In [ ]:

VARIANT_STYLES = {
    "random_clean": ("black", "--"),
    "no_w_clean": ("gray", "-"),
    "optuna_clean": ("purple", "-."),
    "random_noisy_y": ("black", ":"),
    "no_w_noisy_y": ("blue", "-"),
    "optuna_noisy_y": ("purple", ":"),
}

def plot_clean_metrics(fn_name, results_for_fn):
    fig, axes = plt.subplots(1, 3, figsize=(20, 10))

    metrics = [
        ("x_distance_curve", "Mean dist_x"),
        ("clean_curve", "Mean dist_y"),
        ("observed_curve", "Best observed"),
    ]

    for ax, (metric_key, title) in zip(axes, metrics):
        for variant, runs in results_for_fn.items():
            curves = np.array([r[metric_key] for r in runs], dtype=float)
            mean_curve = curves.mean(axis=0)
            std_curve = curves.std(axis=0)

            color, linestyle = VARIANT_STYLES[variant]

            x = np.arange(1, len(mean_curve) + 1)

            ax.plot(
                x,
                mean_curve,
                color=color,
                linestyle=linestyle,
                linewidth=1.0,
                label=variant,
            )

        ax.set_title(title)
        ax.set_xlabel("iteration")
        ax.grid(alpha=0.3)

        # легенда для каждого графика
        ax.legend(fontsize=8)

    axes[0].set_ylabel("value")
    plt.tight_layout()
    plt.show()



def plot_algorithm_choice_map(fn_name, results_for_fn, seed_to_show=0):

    fn = CLEAN_MAP[fn_name]
    bounds = BOUNDS_MAP[fn_name]
    x_opt = TRUE_MINIMA[fn_name]

    x = np.linspace(bounds[0][0], bounds[0][1], 220)
    y = np.linspace(bounds[1][0], bounds[1][1], 220)

    X, Y = np.meshgrid(x, y)

    Z = np.zeros_like(X)

    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            Z[i, j] = fn(np.array([X[i, j], Y[i, j]]))

    fig, axes = plt.subplots(1, len(results_for_fn), figsize=(6 * len(results_for_fn), 5))

    if len(results_for_fn) == 1:
        axes = [axes]

    for ax, (variant, runs) in zip(axes, results_for_fn.items()):

        run = runs[seed_to_show]

        points = np.asarray(run["x_history"], dtype=float)

        clean_values = np.asarray([fn(p) for p in points], dtype=float)

        best_idx = int(np.argmin(clean_values))

        xs = points[:, 0]
        ys = points[:, 1]

        ax.contourf(X, Y, Z, levels=35, cmap="magma")

        ax.plot(
            xs,
            ys,
            linewidth=0.9,
            alpha=0.85,
            color="white",
            zorder=3,
        )

        ax.scatter(
            xs,
            ys,
            s=18,
            alpha=0.75,
            edgecolors="black",
            linewidths=0.3,
            zorder=4,
        )

        dx = xs[1:] - xs[:-1]
        dy = ys[1:] - ys[:-1]

        ax.quiver(
            xs[:-1],
            ys[:-1],
            dx,
            dy,
            angles="xy",
            scale_units="xy",
            scale=1,
            color="black",
            alpha=0.9,
            width=0.0018,
            headwidth=4,
            headlength=5,
            headaxislength=4.5,
            zorder=5,
        )

        ax.scatter(
            xs[0],
            ys[0],
            s=120,
            marker="o",
            edgecolors="cyan",
            facecolors="none",
            linewidths=2,
            label="start",
            zorder=6,
        )

        ax.scatter(
            xs[-1],
            ys[-1],
            s=140,
            marker="X",
            color="lime",
            edgecolors="black",
            linewidths=1,
            label="end",
            zorder=6,
        )

        ax.scatter(
            xs[best_idx],
            ys[best_idx],
            s=180,
            marker="*",
            color="yellow",
            edgecolors="black",
            linewidths=1.2,
            label="best",
            zorder=7,
        )

        ax.scatter(
            x_opt[0],
            x_opt[1],
            s=220,
            marker="P",
            color="red",
            edgecolors="white",
            linewidths=1.2,
            label="optimum",
            zorder=7,
        )

        ax.set_title(variant)
        ax.set_xlabel("x1")
        ax.set_ylabel("x2")
        ax.legend(fontsize=8)

    plt.suptitle(f"{fn_name}: trajectory map")
    plt.tight_layout()
    plt.show()


for fn_name in CLEAN_MAP.keys():
    print("=" * 80)
    print(fn_name)
    print("=" * 80)

    plot_clean_metrics(fn_name, all_results[fn_name])

    plot_algorithm_choice_map(
        fn_name,
        all_results[fn_name],
        seed_to_show=0,
    )


In [ ]:
print("Removed redundant boxplots.")

Removed redundant boxplots.


# 11. Interpretation

Сравниваем внутри каждого метода:

- `random_clean` vs `random_noisy_y`;
- `no_w_clean` vs `no_w_noisy_y`;
- `optuna_clean` vs `optuna_noisy_y`.

Главные вопросы:

1. Насколько шум ухудшает `success_rate_%`?
2. Насколько растут `clean_mean` и `clean_std`?
3. Становится ли TPE ближе к random при шуме?
4. Увеличивается ли variance кривых?

Если шум сильно ухудшает `no_w`, это будет мотивацией для следующего эксперимента с \(w(x)\) при noisy observations.
